# 01 — Data and baselines (MNIST)

Run this notebook first. It downloads MNIST, checks the splits, and computes the
**baselines every method is compared against**:

1. Data and splits
2. PCA spectrum of the 60,000 training images
3. Per-class PCA (one manifold per digit?)
4. Sanity checks of the metrics
5. Reference values in pixel space ($k$-NN accuracy, silhouette)
6. PCA baseline: $R_{NX}$ curves for $q = 2$ and $q = 10$

Outputs: `results/baseline_*.csv`, `figures/baseline_*.png|pdf`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))          # make nldr importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nldr import data, metrics

RESULTS, FIGURES = ROOT / "results", ROOT / "figures"
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)

def save(fig, name):
    """Save a figure as figures/<name>.png and .pdf."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def plot_rnx(curves, ax, title=None):
    """R_NX(K) against log K. curves: label -> (rnx, J)."""
    for label, (rnx, J) in curves.items():
        ax.semilogx(np.arange(1, len(rnx) + 1), rnx, label=f"{label} (J = {J:.3f})")
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set(xlabel="$K$ (log scale)", ylabel="$R_{NX}(K)$", title=title)
    ax.legend(fontsize=8)

In [ ]:
Q_FIG, Q_TAB = 2, 10        # target dimension for figures / tables
K_REPORT = (5, 10, 50)      # K for trustworthiness T(K), continuity C(K)
KNN_K = 5                   # k for k-NN accuracy

## 1. Data and splits

In [ ]:
X, y = data.load_mnist()          # first call downloads ~50 MB, then saved in data/
X_pool, y_pool = X[:60000], y[:60000]
splits = data.make_splits()
s = splits[0]
print("X:", X.shape, "range", X.min(), X.max())
print(f"{len(splits)} splits, n = {data.N_PER_SPLIT} each of tr / val / te")
print("disjoint:", len(set(s.tr) | set(s.val) | set(s.te)) == 3 * data.N_PER_SPLIT)

fig, axes = plt.subplots(3, 10, figsize=(10, 3.4))
for ax, i in zip(axes.ravel(), s.tr[:30]):
    ax.imshow(X[i].reshape(28, 28), cmap="gray_r"); ax.set_title(str(y[i]), fontsize=9); ax.axis("off")
plt.show()

## 2. PCA spectrum (60,000 training images)

Eigenvalues $\lambda_a$ of $\hat\Sigma$ and cumulative explained variance $\sum_{a\le q}\lambda_a / \operatorname{tr}\hat\Sigma$.

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(svd_solver="full").fit(X_pool)
lam = pca_full.explained_variance_
cum = np.cumsum(pca_full.explained_variance_ratio_)
n_for = {p: int(np.searchsorted(cum, p) + 1) for p in (0.8, 0.9, 0.95, 0.99)}
print("components needed:", {f"{int(100*p)}%": q for p, q in n_for.items()})

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].semilogy(np.arange(1, 201), lam[:200], ".-", ms=3)
ax[0].set(xlabel="component $a$", ylabel=r"$\lambda_a$ (log)", title="PCA spectrum (first 200)")
ax[1].plot(np.arange(1, lam.size + 1), cum)
for p, q in n_for.items():
    ax[1].axvline(q, ls=":", c="0.6"); ax[1].text(q, p - 0.06, f"{int(100*p)}%: {q}", fontsize=8)
ax[1].set(xlabel="$q$", ylabel="explained variance", xlim=(0, 400), title="Cumulative explained variance")
save(fig, "baseline_pca_spectrum"); plt.show()

## 3. Per-class PCA

If each digit lies near its own lower-dimensional manifold, one class needs **fewer** components than all classes together.

In [ ]:
rows = []
for digit in range(10):
    cum_c = np.cumsum(PCA(svd_solver="full").fit(X_pool[y_pool == digit]).explained_variance_ratio_)
    rows.append({"digit": digit, "q_90%": int(np.searchsorted(cum_c, 0.9) + 1),
                 "q_95%": int(np.searchsorted(cum_c, 0.95) + 1)})
per_class = pd.DataFrame(rows).set_index("digit")
per_class.loc["all"] = [n_for[0.9], n_for[0.95]]
per_class.to_csv(RESULTS / "baseline_per_class_pca.csv")
per_class.T

## 4. Sanity checks of the metrics

In [ ]:
Xtr = X[s.tr]
rho = metrics.rank_matrix(Xtr)
Y_rand = np.random.default_rng(0).normal(size=(len(s.tr), 2))
J_rand = metrics.auc_log(metrics.rnx_curve(rho, metrics.rank_matrix(Y_rand)))
checks = {
    "R_NX(Y = X) = 1": np.allclose(metrics.rnx_curve(rho, rho), 1.0),
    "random embedding: |J| < 0.02": abs(J_rand) < 0.02,
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
print(f"J of a random embedding: {J_rand:+.4f}")

## 5. Reference values in pixel space

The ceiling for $k$-NN accuracy, and how separated the classes are before any reduction. Mean ± s.d. over the splits.

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for sp in splits:
    rows.append({
        "split": sp.seed,
        "knn_acc_oos": metrics.knn_accuracy_oos(X[sp.tr], y[sp.tr], X[sp.te], y[sp.te], k=KNN_K),
        "knn_acc_cv": metrics.knn_accuracy(X[sp.te], y[sp.te], k=KNN_K),
        "silhouette": silhouette_score(X[sp.te], y[sp.te]),
    })
ref = pd.DataFrame(rows).set_index("split")
ref.to_csv(RESULTS / "baseline_reference_pixel_space.csv")
ref.agg(["mean", "std"]).round(4)

## 6. PCA baseline curves

The benchmark every nonlinear method must beat: $R_{NX}(K)$ of PCA on $X_{\mathrm{te}}$ of split 0.

In [ ]:
Xte, yte = X[s.te], y[s.te]
rho_te = metrics.rank_matrix(Xte)
curves, rows = {}, []
for q in (Q_FIG, Q_TAB):
    Y = PCA(n_components=q, svd_solver="full").fit_transform(Xte)
    res = metrics.evaluate(Xte, Y, yte, Ks=K_REPORT, rho=rho_te)
    curves[f"PCA, q={q}"] = (res.pop("rnx"), res["J"])
    rows.append({"q": q, **{k: round(v, 4) for k, v in res.items()}})
pca_baseline = pd.DataFrame(rows).set_index("q")
pca_baseline.to_csv(RESULTS / "baseline_pca_split0.csv")

fig, ax = plt.subplots(figsize=(6, 4))
plot_rnx(curves, ax, title="PCA baseline, split 0 (test sample)")
save(fig, "baseline_pca_rnx"); plt.show()
pca_baseline